# Main figure code

This notebook preserves the original plotting cells used for the manuscript figures. Only path and variable-preparation cells were added so the code reads from the local `Data/` folder.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import kstest, gaussian_kde, wilcoxon
import statsmodels.api as sm
from pathlib import Path
from mpl_toolkits.basemap import Basemap
from osgeo import gdal
import sys
import types
from scipy.stats import rankdata

spei_path = "Data/"
site_path = "Data/"
drv_path = "Data/"
figout = Path("Figures")
figout.mkdir(exist_ok=True)

def kill_nan(dt):
    a = np.asarray(dt).ravel()
    return list(a[~np.isnan(a)])

def read_img(filename):
    dt = gdal.Open(filename)
    im_width = dt.RasterXSize
    im_height = dt.RasterYSize
    im_bands = dt.RasterCount
    im_geotrans = dt.GetGeoTransform()
    im_proj = dt.GetProjection()
    im_data = dt.ReadAsArray(0, 0, im_width, im_height)
    return im_data, im_width, im_height, im_geotrans, im_proj

gloce_module = types.ModuleType("gloce")
def nanravel(a):
    a = np.asarray(a).ravel()
    return a[~np.isnan(a)]
gloce_module.nanravel = nanravel
sys.modules["gloce"] = gloce_module

pingouin_module = types.ModuleType("pingouin")
def _as_covars(covar):
    if covar is None or covar == []:
        return []
    if isinstance(covar, str):
        return [covar]
    return list(covar)

def partial_corr(data, x, y, covar=None, method="pearson"):
    covars = _as_covars(covar)
    cols = [x, y] + covars
    df = data.loc[:, cols].replace([np.inf, -np.inf], np.nan).dropna()
    xv = df[x].to_numpy(dtype=float)
    yv = df[y].to_numpy(dtype=float)
    if method == "spearman":
        xv = rankdata(xv)
        yv = rankdata(yv)
        z = np.column_stack([rankdata(df[c].to_numpy(dtype=float)) for c in covars]) if covars else None
    else:
        z = df[covars].to_numpy(dtype=float) if covars else None
    if z is not None and z.size:
        X = sm.add_constant(z)
        xv = sm.OLS(xv, X).fit().resid
        yv = sm.OLS(yv, X).fit().resid
    r, p = stats.pearsonr(xv, yv)
    return pd.DataFrame({"n": [len(xv)], "r": [r], "p-val": [p]})

def _df_partial_corr(self, x, y, covar=None, method="pearson"):
    return partial_corr(self, x=x, y=y, covar=covar, method=method)

pingouin_module.partial_corr = partial_corr
pd.DataFrame.partial_corr = _df_partial_corr
sys.modules["pingouin"] = pingouin_module

## Fig. 1

In [ ]:
#vegetation---------------------
#'treeH','tc','lst','spei','et','albedo']
delta_veg_protec=np.load(spei_path+'delta_protect_250613.npy',allow_pickle=True)#(6,260,316),paired sites both within protected areas
delta_veg_unprotec=np.load(spei_path+'delta_unprotect_250613.npy',allow_pickle=True)#(6,260,316),paired sites both without protected areas
pp_veg_protec=np.load(spei_path+'delta_protect_pvalue_250613.npy',allow_pickle=True)#(6,)
pp_veg_unprotec=np.load(spei_path+'delta_unprotect_pvalue_250613.npy',allow_pickle=True)#(6,)

In [ ]:
delta_th=np.append(kill_nan(delta_veg_protec[0]),kill_nan(delta_veg_unprotec[0]))
delta_tc=np.append(kill_nan(delta_veg_protec[1]),kill_nan(delta_veg_unprotec[1]))
data=[delta_th,delta_tc]

In [ ]:
#['treeH','tc_planet']
import gloce as gc
from scipy.stats import gaussian_kde
import matplotlib.colors as mcolors
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
fig = plt.figure(figsize=(12,4)) ##width, height default(8,6)
plt.subplots_adjust(wspace=0.25,hspace=0.02)
#fig.subplots_adjust(wspace=0.2,hspace=0.4, left=None, bottom=None, right=None, top=None)
font = {'family': 'sans-serif',
        'sans-serif': 'Arial',
        'weight': 'normal',
        'size': 18}
plt.rc('font', **font)  # pass in the font dict as kwargs
cl1 = np.array([(47,85,151)])#,(91,155,213)
cl2 = np.array([(22,142,88)])#,(146,208,80)
cl=[cl1/255,cl2/255]
#cl=['C0','salmon']
#colors = ['coral','yellowgreen','green','orange']
#lc3 = ['NF_BB','F_BB','F_NBB','NF_NBB']
#fig, axs=plt.subplots(2,2,sharex=True,sharey=True)
label1=['TH from Potapov $\mathit{et}$ $\mathit{al.}$','TH from Lang $\mathit{et}$ $\mathit{al.}$']
label2=['TC from Reiner $\mathit{et}$ $\mathit{al.}$','TC from Hansen $\mathit{et}$ $\mathit{al.}$']
label=[label1,label2]
tx=['(m)','(%)']
for i in range(2):
    ax = fig.add_subplot(1,2,i+1)
    x=np.linspace(-15,15,1000)
    mean=np.nanmean(data[i])
    delta_rav=gc.nanravel(data[i])
    kenal=gaussian_kde(delta_rav)
    z=kenal.evaluate(x)
    z_mean=kenal.evaluate(mean)
    ax.plot(x,z,lw=3,color=cl[i], zorder=6)#,label=label[i]
    ax.fill_between(x,0,z,facecolor=cl[i],alpha=0.2)
    ax.vlines(mean,0,z_mean,lw=2,ls='--',color=cl[i], zorder=7)
    p=kstest(gc.nanravel(data[i]), 'norm')[1]
    mean = np.nanmean(delta_rav)
    se = stats.sem(delta_rav)     # 标准误差
    ci_low, ci_high = stats.t.interval(0.95, len(delta_rav)-1, loc=mean, scale=se)
    print(mean, ci_low, ci_high)
    print(mean,p,stats.sem(delta_rav),len(delta_rav))
    # 找出CI范围
    #mask = (x >= ci_low) & (x <= ci_high)
    #ax.fill_between(x[mask], 0, z[mask], facecolor="white", alpha=1, zorder=5)
    if p<0.001:
        #ax.text(0.73,0.89-j*0.09, '{:.2f} ***'.format(mean), fontsize=16,transform = ax.transAxes,color=cl[i][j])
        ax.text(0.75,0.89, 'p<0.001', fontsize=14,transform = ax.transAxes,color=cl[i])
    elif p<0.01:
        ax.text(0.75,0.89, 'p<0.01', fontsize=14,transform = ax.transAxes,color=cl[i])
    elif p<0.05:
        ax.text(0.75,0.89, 'p<0.05', fontsize=14,transform = ax.transAxes,color=cl[i])
    else:
        ax.text(0.75,0.89, 'p={:.2f}'.format(p), fontsize=14,transform = ax.transAxes,color=cl[i]) 

    if i ==0:
        ax.vlines(0,0,0.3,lw=2,ls='--',color='black')
        ax.set_ylim(0,0.3)
        ax.set_yticks(np.arange(0,0.31,0.1))
        ax.set_xlim(-5,5)
        ax.set_xticks(np.arange(-5,5.1,2.5))
        ax=plt.gca()
        ax.yaxis.set_ticks_position('left')
        ax.spines['left'].set_position(('data',-5.3))
        ax.xaxis.set_ticks_position('bottom')
        ax.spines['bottom'].set_position(('data',-0.04))
        ax.set_xlabel('$\Delta$Tree height (m)',labelpad=10)
    else:
        ax.vlines(0,0,0.08,lw=2,ls='--',color='black')
        ax.set_ylim(0,0.08)
        ax.set_yticks(np.arange(0,0.081,0.02))
        ax.set_xlim(-15,15)
        ax.set_xticks(np.arange(-15,15.1,5))
        ax=plt.gca()
        ax.yaxis.set_ticks_position('left')
        ax.spines['left'].set_position(('data',-15.8))
        ax.xaxis.set_ticks_position('bottom')
        ax.spines['bottom'].set_position(('data',-0.0105))
        ax.set_xlabel('$\Delta$Tree cover (%)',labelpad=10)
    ax.legend(fontsize=14,loc='upper left',frameon=False,handlelength=1)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    #ax.spines['bottom'].set_visible(False)
    ax.set_ylabel('Probability density',labelpad=10)
    
    ax.xaxis.label.set_size(18)
    ax.xaxis.set_tick_params(width=2)

    ax.tick_params(labelsize=14)
    ax.yaxis.label.set_size(18)
    ax.yaxis.set_tick_params(width=2)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
#添加箱线图----------------------------
    if i ==0:
        ax1=fig.add_axes([0.122,-0,0.35,0.12])#左，底，宽，高
        ax1.set_xlim(-5,5)
        ax1.set_xticks(np.arange(-5,5,2.5))
        ax1.axis('off')
    else:
        ax1=fig.add_axes([0.55,-0,0.35,0.12])#左，底，宽，高
        ax1.set_xlim(-15,15)
        ax1.set_xticks(np.arange(-15,15,5))
        ax1.axis('off')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['left'].set_visible(False)
    ax1.tick_params(bottom=False,top=False, left=False, right=False)#隐藏刻度线
    #ax.spines['bottom'].set_visible(False)
    bplot=ax1.boxplot(data[i],
                      vert=False,
                      whis=(10,90),                
                      widths=0.3,
                      patch_artist=True,
                      showmeans=False,
                      meanprops = {'marker':'o','markerfacecolor':cl[i],"markeredgecolor":cl[i],"markersize":15,"alpha":0}, # 设置均值点的属性，点的形状、填充色
                      medianprops={'linewidth':'0.5',"color":cl[i],"alpha":0},
                      boxprops={"facecolor": 'none', "edgecolor": cl[i],"linewidth":2,"alpha":1},
                      capprops=None,
                      whiskerprops={'linewidth':'2','color':cl[i]},
                      showcaps=False,# 是否显示箱线图顶端和末端的两条线，默认显示；
                      showfliers = False,
                      flierprops = {'marker':'+','markersize':'3','markeredgecolor':cl[i],'color':cl[i]})
    
#rect = ax.patch
#rect.set_facecolor((202/255,233/255,246/255))
#fig.savefig("E:/iCloudDrive/博后工作/文章/Elephant_disturb/Fig/reviseV2/Fig2_c-d.tif", dpi=600, bbox_inches = 'tight')

In [ ]:
#加载数据
data = np.load(spei_path+"elephant_effect_results.npz", allow_pickle=True)
estimates_tc = data["estimates_tc"].item()
cis_tc       = data["cis_tc"].item()
estimates_th = data["estimates_th"].item()
cis_th       = data["cis_th"].item()
print(estimates_tc, cis_tc)
#means_tc, xerr_tc = prep_data(estimates_tc, cis_tc)
#means_th, xerr_th = prep_data(estimates_th, cis_th)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
# ====== 假设你已经定义好 estimates_tc / cis_tc / estimates_th / cis_th ======
label_to_key = {
    "Matching": "ATT-Match",
    "IPW": "ATE-IPW",
    "Standardization": "ATE-Gcomp",
    "DR": "ATE-DR",}
labels = list(label_to_key.keys())
y = np.arange(len(labels))
#y = np.array([1, 1.8, 2.6, 3.4])
def prep_data(estimates, cis):
    means, xerr_lo, xerr_hi = [], [], []
    for lab in labels:
        key = label_to_key[lab]
        est = estimates.get(key, np.nan)
        lo, hi = cis.get(key, (np.nan, np.nan))
        left  = max(est - lo, 0.0) if np.isfinite(est) and np.isfinite(lo) else 0.0
        right = max(hi - est, 0.0) if np.isfinite(est) and np.isfinite(hi) else 0.0
        means.append(est)
        xerr_lo.append(left)
        xerr_hi.append(right)
    return np.array(means), [xerr_lo, xerr_hi]

means_tc, xerr_tc = prep_data(estimates_tc, cis_tc)
means_th, xerr_th = prep_data(estimates_th, cis_th)
# ====== 画图 ======
fig, axes = plt.subplots(1, 2, figsize=(12,4), sharey=True)
plt.subplots_adjust(wspace=0.2,hspace=0.02)
font = {'family': 'sans-serif', 'sans-serif': 'Arial', 'weight': 'normal', 'size': 16}
plt.rc('font', **font)

c_th_marker = (47/255, 85/255, 151/255)  
c_th_error  = (176/255, 191/255, 216/255) 
c_tc_marker = (22/255, 142/255, 88/255) 
c_tc_error  = (177/255, 217/255, 199/255)
# ---------- 左图：TH ----------
ax1 = axes[0]
#ax.errorbar(xm, ym, yerr=se, fmt='o',mfc='white', mec=color, ecolor='lightgray',ms=6, lw=1, capsize=0, zorder=3)
ax1.errorbar(means_th, y, xerr=xerr_th, fmt="o", mfc=c_th_marker, mec=c_th_marker, ms=10, ecolor=c_th_error, lw=5, capsize=0, barsabove=True)
ax1.axvline(0, linestyle="--", color="black", linewidth=2)
ax1.set_xlim(-2,4)
ax1.set_xticks(np.arange(-2,4.01,2))
ax1.set_xlabel("Effects of elephants on tree height (m)", labelpad=10)
ax1.tick_params(labelsize=18)
ax1.xaxis.label.set_size(18)
ax1.xaxis.set_tick_params(width=1.5)
#ax1.spines['top'].set_visible(False)
#ax1.spines['right'].set_visible(False)
#ax1.spines['left'].set_visible(False)
ax1.spines['top'].set_linewidth(1.5)
ax1.spines['right'].set_linewidth(1.5)
ax1.spines['left'].set_linewidth(1.5)
ax1.spines['bottom'].set_linewidth(1.5)
ax1.set_ylim(min(y)-0.5, max(y)+0.5)
# ---------- 右图：TC ----------
ax2 = axes[1]
ax2.errorbar(means_tc, y, xerr=xerr_tc, fmt="o",mfc=c_tc_marker, mec=c_tc_marker, ms=10, ecolor=c_tc_error, lw=5, capsize=0, barsabove=True)
ax2.axvline(0, linestyle="--", color="black", linewidth=2)
ax2.set_xlim(-5,10)
ax2.set_xticks(np.arange(-5,10.01,5))
ax2.set_yticks(y)
ax2.set_yticklabels(labels)
ax2.set_xlabel("Effects of elephants on tree cover (%)", labelpad=10)
ax2.tick_params(labelsize=18)
ax2.xaxis.label.set_size(18)
ax2.xaxis.set_tick_params(width=1.5)
#ax2.spines['top'].set_visible(False)
#ax2.spines['right'].set_visible(False)
#ax2.spines['left'].set_visible(False)
ax2.spines['top'].set_linewidth(1.5)
ax2.spines['right'].set_linewidth(1.5)
ax2.spines['left'].set_linewidth(1.5)
ax2.spines['bottom'].set_linewidth(1.5)
ax2.set_ylim(min(y)-0.5, max(y)+0.5)

## Fig. 2

### Fig. 2a,b

In [ ]:
val=np.load(spei_path+'ele_veg_val_250613.npy')
std=np.load(spei_path+'ele_veg_std_250613.npy')
dens=np.load(spei_path+'ele_veg_dens_250613.npy')

In [ ]:
dens_site=pd.read_csv(spei_path+"contry_level_density.csv")
dens_site.info()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from scipy.stats import linregress
from sklearn.linear_model import TheilSenRegressor
import pingouin as pg
import statsmodels.api as sm
from scipy.stats import gaussian_kde
import warnings
from scipy.stats import wilcoxon
def annotate_wilcoxon_mean_p(ax, y, color, pos=(0.85, 0.85), fontsize=14):
    """
    在轴的固定位置标注 Wilcoxon 符号秩检验（相对 0）的双尾 p 值。
    """
    y = np.asarray(y, float)
    y = y[np.isfinite(y)]

    # 严格处理 0 值与样本量；'wilcox' 会丢弃 0 差值
    y_nonzero = y[y != 0]
    if y_nonzero.size < 5:  # 太少就不标
        return
    try:
        stat, p = wilcoxon(y_nonzero, alternative='two-sided',
                           zero_method='wilcox', mode='auto')
    except Exception:
        return
    if p < 1e-3:
        txt = r'$p_{\mathrm{mean}}<0.001$'
    elif p < 1e-2:
        txt = r'$p_{\mathrm{mean}}<0.01$'
    elif p < 5e-2:
        txt = r'$p_{\mathrm{mean}}<0.05$'
    else:
        txt = rf'$p_{{\mathrm{{mean}}}}={p:.2f}$'

    ax.text(pos[0], pos[1], txt,
            transform=ax.transAxes, ha='right', va='top',
            fontsize=14, color=color, zorder=10)

def permutation_p(x, y, n_perm=5000, seed=None):
    """
    双尾置换检验 slope 的显著性
    """
    x = np.asarray(x)
    y = np.asarray(y)
    rng = np.random.RandomState(seed)
    slope0, _, _, _, _ = linregress(x, y)
    count = 0
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        slope_perm, _, _, _, _ = linregress(x, y_perm)
        if abs(slope_perm) >= abs(slope0):
            count += 1
    return (count + 1) / (n_perm + 1)

def linear_fit_ci_on_binned(ax, xm, ym, color, label=None, count=None,use_pred_interval=False, n_perm=5000, seed=42):
    """
    线性回归 + 95%CI + 仅在显著时绘线
    p 值使用双尾置换检验，稳健且不依赖正态性假设。
    """
    ok = np.isfinite(xm) & np.isfinite(ym)
    xm, ym = xm[ok], ym[ok]
    w = None
    if count is not None:
        w_full = np.asarray(count)[ok]
        w = np.where(np.isfinite(w_full) & (w_full > 0), w_full, np.nan)
        if np.all(~np.isfinite(w)):
            w = None
    if xm.size < 3:
        return None, None
    X = sm.add_constant(xm)
    model = sm.WLS(ym, X, weights=w).fit() if w is not None else sm.OLS(ym, X).fit()
    # ---- 置换检验 p 值（替代 model.pvalues[1]）----
    p_slope = permutation_p(xm, ym, n_perm=n_perm, seed=seed)
    # ---- 若不显著：跳过绘图 ----
    if p_slope >= 0.05:
        return r2_score(ym, model.predict(X)), p_slope
    # ---- 绘制拟合线 + CI ----
    xmin, xmax = float(np.min(xm)), float(np.max(xm))
    xx = np.linspace(xmin, xmax, 400)
    XX = sm.add_constant(xx)
    pred = model.get_prediction(XX).summary_frame(alpha=0.05)
    y_pred = pred['mean'].values
    ci_low  = pred['obs_ci_lower'].values if use_pred_interval else pred['mean_ci_lower'].values
    ci_high = pred['obs_ci_upper'].values if use_pred_interval else pred['mean_ci_upper'].values
    ax.plot(xx, y_pred, color=color, lw=2.5, label=label)
    ax.fill_between(xx, ci_low, ci_high, color=color, edgecolor="none", alpha=0.2)
    # ---- 文本 p 值注释 ----
    if p_slope < 1e-3:
        txt = r'$p_{\mathrm{fit}}<0.001$'
    elif p_slope < 1e-2:
        txt = r'$p_{\mathrm{fit}}<0.01$'
    elif p_slope < 5e-2:
        txt = r'$p_{\mathrm{fit}}<0.05$'
    else:
        txt = rf'$p_{{\mathrm{{mean}}}}={p_slope:.2f}$'

    ax.text(0.85, 0.92, txt,transform=ax.transAxes,ha='right', va='top',fontsize=14, color=color)
    return r2_score(ym, model.predict(X)), p_slope

def draw_side_densities(ax, y_groups, colors, x_pos='right', pad_frac=0.02,
                        width_frac=0.10, n_pts=300, lw=1.5,
                        show_ci=True, ci_kind='normal', ci_alpha=1.0,  # ci_kind: 'normal' | 'none'
                        ci_edgecolor=None, ci_edgewidth=1.0):
    """
    在主坐标内靠右侧绘制每组 y 的竖向 KDE 密度条，并画均值线（与主图共用 y 轴）。
    额外：为“均值的 95% 置信区间”绘制白色填充带（与局部密度对齐）。
    参数：
      show_ci      : 是否绘制 95% CI（均值）
      ci_kind      : 'normal' 使用 μ±1.96·SE（SE=s/√n）的正态近似；'none' 不画
      ci_alpha     : CI 白色填充不透明度（1=纯白）
      ci_edgecolor : CI 外边框颜色（默认与组颜色一致）
      ci_edgewidth : CI 外边框宽度
    """
    # --- 当前轴范围 ---
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    xspan = max(1e-12, xmax - xmin)
    # 贴靠位置
    if x_pos == 'right':
        x_right = xmax - pad_frac * xspan
    else:
        x_right = float(x_pos)
    # 最大条宽（横向）
    width = width_frac * xspan
    # 竖向网格
    yy = np.linspace(ymin, ymax, int(max(100, n_pts)))
    for idx, yg in enumerate(y_groups):
        if yg is None:
            continue
        y = np.asarray(yg, float)
        y = y[np.isfinite(y)]
        if y.size < 3:
            continue
        c = colors[idx % len(colors)]
        # KDE & 归一到固定“条宽”
        kde = gaussian_kde(y)
        dens = kde(yy)
        dens_scaled = dens / dens.max() * width if dens.max() > 0 else np.zeros_like(dens)
        # --- 密度填充+外轮廓 ---
        ax.fill_betweenx(yy, x_right - dens_scaled, x_right, color=c, alpha=0.20, lw=0, zorder=1)
        ax.plot(x_right - dens_scaled, yy, color=c, lw=lw, zorder=3)
        # --- 均值线（与局部密度对齐的“短横线”） ---
        mu = float(np.nanmean(y))
        i_mu = int(np.argmin(np.abs(yy - mu)))
        local_w = dens_scaled[i_mu]
        ax.hlines(mu, x_right - local_w, x_right, linestyle='--', colors=c, lw=lw, zorder=4)
        # --- 95% CI（均值）白色填充带：与局部密度对齐 ---
        """
        if show_ci and ci_kind != 'none':
            # 正态近似：mu ± 1.96*SE
            if ci_kind == 'normal':
                s = float(np.nanstd(y, ddof=1)) if y.size > 1 else 0.0
                se = s / np.sqrt(y.size) if y.size > 0 else 0.0
                lo = mu - 1.96 * se
                hi = mu + 1.96 * se
            else:
                lo, hi = mu, mu  # 兜底（不应该走到这里）
            lo = max(lo, ymin)
            hi = min(hi, ymax)
            if hi > lo:  # 有有效区间才画
                mask_ci = (yy >= lo) & (yy <= hi)
                if np.any(mask_ci):
                    x_left_ci = x_right - dens_scaled[mask_ci]
                    yy_ci = yy[mask_ci]
                    # 先白色填充，覆盖密度内部
                    ax.fill_betweenx(yy_ci, x_left_ci, np.full_like(yy_ci, x_right),color='white', alpha=ci_alpha, lw=0, zorder=3)
                    """

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
fig = plt.figure(figsize=(12,4)) ##width, height default(8,6)
fig.subplots_adjust(wspace=0.25,hspace=0.02, left=None, bottom=None, right=None, top=None)
font = {'family': 'sans-serif',
        'sans-serif': 'Arial',
        'weight': 'normal',
        'size': 16}
plt.rc('font', **font)  # pass in the font dict as kwargs
cl1 = [(47/255,85/255,151/255),(22/255,142/255,88/255)]
cl2  = [(150/255,176/255,222/255), (188/255,222/255,207/255)]
x1=np.linspace(0,1.3,1000)
x2=np.linspace(0,1.3,1000)
zero=np.zeros((1000))

for i in range(2):
    ax=fig.add_subplot(1,2,i+1)
    #ax.errorbar(dens[i],val[i],yerr=[std[i],std[i]],fmt='o',ecolor='lightgray',elinewidth=1,capthick=0,ms=8,mfc='none',mec=cl1[i],mew=2,capsize=0)
    ax.errorbar(dens[i],val[i],yerr=[std[i],std[i]],fmt='o',ecolor=cl2[i], mfc='white', mec=cl1[i], ms=6, lw=1, capsize=0, zorder=3)
    plt.plot(x2,zero,color='black',linestyle='--',linewidth=1.5)
    linear_fit_ci_on_binned(ax, dens[i],val[i], color=cl1[i])
    #ax.vlines(30,-5,10,lw=1,ls='--',color='grey')
    plt.xlabel('Elephant density (N km$^{-2}$)',fontsize=18,labelpad=10)
    ax.set_xlim(0,1.25)
    ax.set_xticks(np.arange(0,1.25,0.3))
    ax=plt.gca()
    ax.yaxis.set_ticks_position('left')
    ax.spines['left'].set_position(('data',-0.03))
    if i==0:
        ax.set_ylim(-5,15)
        ax.set_yticks(np.arange(-5,15.01,5))
        ax.set_ylabel(''r'$\Delta$Tree height'" (m)",fontsize=18,labelpad=10)
    if i==1:
        ax.set_ylim(-20,40)
        ax.set_yticks(np.arange(-20,40.01,20))
        ax.set_ylabel(''r'$\Delta$Tree cover'" (%)",fontsize=18,labelpad=10)
    ax.xaxis.label.set_size(18)
    ax.yaxis.label.set_size(18)
    ax.tick_params(labelsize=14)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.xaxis.set_tick_params(width=2)
    ax.yaxis.set_tick_params(width=2)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
    # 第 i 个子图使用 dens_site 中的原始数据
    if i == 0:
        y_groups = [dens_site["TH"].values]   # 图1: ΔTree Height
    elif i == 1:
        y_groups = [dens_site["TC"].values]   # 图2: ΔTree Cover
    draw_side_densities(ax,y_groups=y_groups,colors=[cl1[i]],x_pos='right', pad_frac=0.0,width_frac=0.08,n_pts=300,lw=2)
    if i == 0:
        y_for_test = dens_site["TH"].values
    else:
        y_for_test = dens_site["TC"].values
    annotate_wilcoxon_mean_p(ax, y_for_test, color=cl1[i], pos=(0.85, 0.85), fontsize=14)

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.font_manager")
#fig.savefig("E:/iCloudDrive/博后工作/文章/Elephant_disturb/Fig/reviseV2/Fig4.tif", dpi=600, bbox_inches = 'tight')

### Fig. 2c,d

In [ ]:
df=pd.read_csv(spei_path+"patch_level_density_both_protected.csv")
df.info()

In [ ]:
del_th=df['TH']
del_tc=df['TC']
ele_dens=df['Ele_Dens']

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from scipy.stats import gaussian_kde
import statsmodels.api as sm
from sklearn.metrics import r2_score
import warnings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

fig = plt.figure(figsize=(12, 4))
fig.subplots_adjust(wspace=0.3, hspace=0, left=None, bottom=None, right=None, top=None)
font = {'family': 'sans-serif',
        'sans-serif': 'Arial',
        'weight': 'normal',
        'size': 16}
plt.rc('font', **font)
# ========= 读取数据 =========
th   = del_th
tc   = del_tc
dens = ele_dens

def _safe_nanstd(a):
    a = np.asarray(a, float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return np.nan
    m = np.nanmean(a)
    return float(np.sqrt(np.nanmean((a - m) ** 2)))

def bin_stats_adaptive(x, y, min_pts=20, min_bins=5, max_bins=40):
    """
    等频自适应分箱：尽量让每箱样本数 >= min_pts
    返回 (xm, ym, se, count, edges)
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    n = x.size
    if n < max(3, min_pts):
        return np.array([]), np.array([]), np.array([]), np.array([]), np.array([])

    # 目标箱数：大约 n / min_pts，并截在 [min_bins, max_bins] 内
    nbin = int(np.clip(n // max(1, min_pts), min_bins, max_bins))
    uniq = np.unique(x[~np.isnan(x)])
    nbin = int(np.clip(nbin, 1, max(1, uniq.size - 1)))

    q = np.linspace(0, 1, nbin + 1)
    edges = np.quantile(x, q)

    # 防止相同边界导致 digitize 异常
    for k in range(1, edges.size):
        if edges[k] <= edges[k - 1]:
            edges[k] = np.nextafter(edges[k - 1], np.inf)

    idx = np.digitize(x, edges) - 1
    xm, ym, se, count = [], [], [], []
    for k in range(nbin):
        m = (idx == k)
        nk = int(m.sum())
        if nk == 0:
            continue
        xm.append(float(np.nanmean(x[m])))
        ym.append(float(np.nanmean(y[m])))
        std_k = _safe_nanstd(y[m])
        se.append(float(std_k / np.sqrt(nk)) if nk > 1 else np.nan)
        count.append(nk)

    return np.array(xm), np.array(ym), np.array(se), np.array(count), edges

def draw_side_densities(ax, y_groups, colors, x_pos='right', pad_frac=0.02,
                        width_frac=0.10, n_pts=300, lw=1.5):
    """
    在主坐标内靠右侧绘制每组 y 的竖向 KDE 密度条，并画均值线（与主图共用 y 轴）。
    - y_groups: list[np.ndarray]，每个元素是一组 y 的原始样本
    - colors  : list，与 y_groups 等长（或可循环使用）
    - x_pos   : 'right' 用当前坐标右边界；也可传 float 指定精确 x 位置
    - pad_frac: 若 x_pos='right'，密度条左移的边距占 x 轴跨度的比例
    - width_frac: 密度条最大宽度占 x 轴跨度比例
    """
    # 当前轴范围
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    xspan = max(1e-12, xmax - xmin)

    # 贴靠位置
    if x_pos == 'right':
        x_right = xmax - pad_frac * xspan
    else:
        x_right = float(x_pos)

    # 最大横向宽度
    width = width_frac * xspan

    # 竖向网格
    yy = np.linspace(ymin, ymax, int(max(100, n_pts)))

    for idx, yg in enumerate(y_groups):
        if yg is None:
            continue
        y = np.asarray(yg, float)
        y = y[np.isfinite(y)]
        if y.size < 3:
            continue

        c = colors[idx % len(colors)]

        # KDE & 归一到固定“条宽”
        kde = gaussian_kde(y)
        dens = kde(yy)
        if dens.max() > 0:
            dens_scaled = dens / dens.max() * width
        else:
            dens_scaled = np.zeros_like(dens)

        # 填充 + 外轮廓
        ax.fill_betweenx(yy, x_right - dens_scaled, x_right,
                         color=c, alpha=0.20, lw=0, zorder=1)
        ax.plot(x_right - dens_scaled, yy, color=c, lw=lw, zorder=3)

        # 均值线（短横线）
        mu = float(np.nanmean(y))
        i_mu = np.argmin(np.abs(yy - mu))
        local_w = dens_scaled[i_mu]
        ax.hlines(mu, x_right - local_w, x_right,
                  linestyle='--', colors=c, lw=lw, zorder=4)
from scipy.stats import wilcoxon, linregress

def annotate_wilcoxon_mean_p(ax, y, color, pos=(0.85, 0.82), fontsize=14):
    """
    在轴的固定位置标注 Wilcoxon 符号秩检验（相对 0）的双尾 p_mean。
    """
    y = np.asarray(y, float)
    y = y[np.isfinite(y)]

    # Wilcoxon 会丢掉 0 差值，这里先去 0
    y_nonzero = y[y != 0]
    if y_nonzero.size < 5:  # 样本太少就不标
        return

    try:
        stat, p = wilcoxon(y_nonzero, alternative='two-sided',
                           zero_method='wilcox', mode='auto')
    except Exception:
        return

    if p < 1e-3:
        txt = r'$p_{\mathrm{mean}}<0.001$'
    elif p < 1e-2:
        txt = r'$p_{\mathrm{mean}}<0.01$'
    elif p < 5e-2:
        txt = r'$p_{\mathrm{mean}}<0.05$'
    else:
        txt = rf'$p_{{\mathrm{{mean}}}}={p:.2f}$'

    ax.text(pos[0], pos[1], txt,transform=ax.transAxes, ha='right', va='top',
            fontsize=fontsize, color=color, zorder=10)

def permutation_p(x, y, n_perm=5000, seed=42):
    """
    线性回归 slope 的双尾置换检验 p_fit（比直接用 OLS p 更稳健一点）。
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]

    if x.size < 3:
        return np.nan

    rng = np.random.RandomState(seed)
    slope0, _, _, _, _ = linregress(x, y)

    count = 0
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        slope_perm, _, _, _, _ = linregress(x, y_perm)
        if abs(slope_perm) >= abs(slope0):
            count += 1

    # +1 / +1 做一个小的连续性校正
    return (count + 1) / (n_perm + 1)

def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=50):
    new_cmap = colors.LinearSegmentedColormap.from_list(
        "trunc({n},{a:.2f},{b:.2f})".format(n=cmap.name, a=minval, b=maxval),
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap
trunc_cl1 = truncate_colormap(plt.get_cmap("Greys"), 0.1, 0.8)
trunc_cl2 = truncate_colormap(plt.get_cmap("Greys"), 0.1, 0.8)
cmaps = [trunc_cl1, trunc_cl2]
# 线条颜色：蓝 / 绿
cl1  = [(47/255,85/255,151/255), (22/255,142/255,88/255)]
cl2  = [(150/255,176/255,222/255), (188/255,222/255,207/255)]
# 点的密度 colormap
pairs = [(th, r'$\Delta$Tree height$\,(\mathrm{m})$'),
         (tc, r'$\Delta$Tree cover$\,(\%)$')]
im_last = None  # 用于右图 colorbar
for i, (y_raw, ylabel) in enumerate(pairs, start=1):
    ax = fig.add_subplot(1, 2, i)
    # ---------- 数据准备 ----------
    x_arr = np.asarray(dens)
    y_arr = np.asarray(y_raw)
    ok = np.isfinite(x_arr) & np.isfinite(y_arr)
    x_arr = x_arr[ok]
    y_arr = y_arr[ok]
    # ---------- 自适应分箱：计算 bin 均值 ± SE ----------
    xm, ym, se, count, edges = bin_stats_adaptive(x_arr, y_arr,min_pts=15,min_bins=5,max_bins=50)
    # ---------- KDE 点密度着色 ----------
    xy = np.vstack([x_arr, y_arr])
    z = gaussian_kde(xy)(xy)          # 每个点的密度
    idx = z.argsort()                 # 从低到高排序避免遮挡
    x_s, y_s, z_s = x_arr[idx], y_arr[idx], z[idx]
    z_stretch = z_s ** 0.6
    im = ax.scatter(x_s, y_s, c=z_s, cmap=cmaps[i-1],s=10)#, edgecolors=cl2[i-1]
    # ---------- 线性回归（OLS） ----------
    X = sm.add_constant(x_arr)
    model = sm.OLS(y_arr, X).fit()
    xmin, xmax = float(np.min(x_arr)), float(np.max(x_arr))
    xx = np.linspace(xmin, 3.3, 400)
    XX = sm.add_constant(xx)
    sf = model.get_prediction(XX).summary_frame(alpha=0.05)
    y_pred = sf['mean'].values
    ci_low = sf['mean_ci_lower'].values
    ci_high = sf['mean_ci_upper'].values
    # R² & p 值
    y_hat = model.predict(X)
    r2 = r2_score(y_arr, y_hat)
    #p_slope = float(model.pvalues[1])
    p_fit = permutation_p(x_arr, y_arr, n_perm=5000, seed=42)
    # ---------- 画零线 + 回归线 + CI ----------
    #ax.plot(xx, np.zeros_like(xx), linestyle='--',linewidth=1.5, color='black')
    ax.plot(xx, y_pred, color=cl1[i-1], lw=2.5)
    ax.fill_between(xx, ci_low, ci_high,color=cl1[i-1], alpha=0.18, linewidth=0)
    # ---------- 画 bin 均值 ± SE ----------
    if xm.size > 0:
        ax.errorbar(xm, ym, yerr=se,fmt='o', mfc='white',mec=cl1[i-1],ecolor=cl2[i-1],elinewidth=1,
                    capsize=0,ms=6,label='Bin mean ± SE')
        #ax.errorbar(dens[i],val[i],yerr=[std[i],std[i]],fmt='o',ecolor='lightgray', mfc='white', mec=cl1[i], ms=6, lw=1, capsize=0, zorder=3)
    # ---------- 文本标注（右上） ----------
    y_max, y_min = np.nanmax(y_arr), np.nanmin(y_arr)
    yr = max(1e-9, y_max - y_min)
    x_text = xmin + 0.70 * (xmax - xmin)
    y_text1 = y_max - 0.12 * yr
    y_text2 = y_max - 0.24 * yr
    #ax.text(x_text, y_text1, f'$R^2={r2:.2f}$',fontsize=14, color=cl1[i-1])
    # ---------- 文本标注（右上：p_fit） ----------
    if np.isfinite(p_fit):
        if p_fit < 1e-3:
            txt_fit = r'$p_{\mathrm{fit}}<0.001$'
        elif p_fit < 1e-2:
            txt_fit = r'$p_{\mathrm{fit}}<0.01$'
        elif p_fit < 5e-2:
            txt_fit = r'$p_{\mathrm{fit}}<0.05$'
        else:
            txt_fit = rf'$p_{{\mathrm{{fit}}}}={p_fit:.2f}$'

        ax.text(0.6, 0.9, txt_fit,fontsize=14, transform=ax.transAxes, color=cl1[i-1])
    # ---------- 轴标签 & 范围 ----------
    ax.set_xlabel('Elephant density (N km$^{-2}$)', fontsize=18, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=18, labelpad=10)
    ax.set_xlim(0, 3.8)
    ax.set_xticks(np.arange(0, 3.8, 1))
    xx = np.linspace(xmin, 3.8, 400)
    ax.plot(xx, np.zeros_like(xx), linestyle='--',linewidth=1.5, color='black')
    ax=plt.gca()
    ax.yaxis.set_ticks_position('left')
    ax.spines['left'].set_position(('data',-0.09))
    if i == 1:
        ax.set_ylim(-3, 6)
        ax.set_yticks(np.arange(-3, 6.01, 3))
    else:
        ax.set_ylim(-15, 20)
        ax.set_yticks(np.arange(-15, 20.01, 5))
    # ---------- 外观 ----------
    ax.tick_params(labelsize=14)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
    ax.xaxis.set_tick_params(width=2)
    ax.yaxis.set_tick_params(width=2)
    # ---------- 右侧竖向 density ----------
    draw_side_densities(ax,y_groups=[y_arr], colors=[cl1[i-1]],x_pos='right',pad_frac=0.0,width_frac=0.08,n_pts=300,lw=2)
    # ---------- p_mean（Wilcoxon：ΔTH/ΔTC 是否整体偏离 0） ----------
    annotate_wilcoxon_mean_p(ax,y_arr,color=cl1[i-1],pos=(0.85, 0.87),fontsize=14)
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib.font_manager")
plt.show()

## Fig. 3

In [ ]:
#vegetation and lst1516----------
#'treeH','tc','lst','spei','et','albedo']
delta_veg_protec=np.load(spei_path+'delta_protect_250613.npy',allow_pickle=True)#(6,260,316)
delta_veg_unprotec=np.load(spei_path+'delta_unprotect_250613.npy',allow_pickle=True)#(6,260,316)
pp_veg_protec=np.load(spei_path+'delta_protect_pvalue_250613.npy',allow_pickle=True)#(6,)
pp_veg_unprotec=np.load(spei_path+'delta_unprotect_pvalue_250613.npy',allow_pickle=True)#(6,)

#lst1214---------------------
delta_lst1214_protec=np.load(spei_path+'delta_lst1214_protect_250613.npy',allow_pickle=True)#(1,260,316)
delta_lst1214_unprotec=np.load(spei_path+'delta_lst1214_unprotect_250613.npy',allow_pickle=True)#(1,260,316)
pp=np.load(spei_path+'delta_lst1214_protect_pvalue_250613.npy',allow_pickle=True)#(2,)
pp_lst1214_protec=pp[0]
pp_lst1214_unprotec=pp[1]

In [ ]:
def kill_nan(dt):
    a=dt.ravel()
    a=a[~np.isnan(a)]
    a=list(a)
    return a
delta_lst1516=np.append(kill_nan(delta_veg_protec[2]),kill_nan(delta_veg_unprotec[2]))
delta_lst1214=np.append(kill_nan(delta_lst1214_protec),kill_nan(delta_lst1214_unprotec))
df_delta=[delta_lst1516,delta_lst1214]

In [ ]:
#-*- coding:utf-8 –*-
import gloce as gc
from scipy.stats import kstest
from scipy.stats import gaussian_kde
import matplotlib.colors as mcolors
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
fig = plt.figure(figsize=(12,4)) ##width, height default(8,6)
plt.subplots_adjust(wspace=0.25,hspace=0.02)
#fig.subplots_adjust(wspace=0.2,hspace=0.4, left=None, bottom=None, right=None, top=None)
font = {'family': 'sans-serif',
        'sans-serif': 'Arial',
        'weight': 'normal',
        'size': 14}
plt.rc('font', **font)  # pass in the font dict as kwargs
#pp=[p_delta_LA[1],p_delta_LA[0]]
#pp1=[p_delta_LC[1],p_delta_LC[0]]
cl1= np.array([(211,145,161)])
cl2= np.array([(231,195,204)])
cl=[cl1/255,cl1/255]
#cl2=['C0','salmon']
label1=['Drought area']
label2=['Drought area']
label=[label1,label2]
#colors = ['coral','yellowgreen','green','orange']
#lc3 = ['NF_BB','F_BB','F_NBB','NF_NBB']
#fig, axs=plt.subplots(2,2,sharex=True,sharey=True)
for i in range(2):
    ax = fig.add_subplot(1,2,i+1)
    x=np.linspace(-2,2,1000)
    mean=np.nanmean(df_delta[i])
    delta_rav=gc.nanravel(df_delta[i])
    kenal=gaussian_kde(delta_rav)
    z=kenal.evaluate(x)
    z_mean=kenal.evaluate(mean)
    ax.plot(x,z,lw=3,color=cl[0],label=label[i])
    ax.fill_between(x,0,z,facecolor=cl[i],alpha=0.2)
    ax.vlines(mean,0,z_mean,lw=2,ls='--',color=cl[0])
    ax.vlines(0,0,0.8,lw=2,ls='--',color='black')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    p=kstest(gc.nanravel(df_delta[i]), 'norm')[1]
    print(mean,p,len(delta_rav))
    if p<0.001:
        #ax.text(0.73,0.89-j*0.09, '{:.2f} ***'.format(mean), fontsize=16,transform = ax.transAxes,color=cl[i][j])
        ax.text(0.75,0.89, 'p<0.001', fontsize=14,transform = ax.transAxes,color=cl[0])
    elif p<0.01:
        ax.text(0.75,0.89, 'p<0.01', fontsize=14,transform = ax.transAxes,color=cl[0])
    elif p<0.05:
        ax.text(0.75,0.89, 'p<0.05', fontsize=14,transform = ax.transAxes,color=cl[0])
    else:
        ax.text(0.75,0.89, 'p={:.2f}'.format(p), fontsize=14,transform = ax.transAxes,color=cl[0]) 

    if i ==0:
        ax.set_xlabel('$\Delta$LST during 2015-2016 (°C)',labelpad=10)
    else:
        ax.set_xlabel('$\Delta$LST during 2012-2014 (°C)',labelpad=10)

    #ax.text(-0.15,1, '(c)', transform = ax.transAxes,color='black',fontsize=20)
    ax=plt.gca()
    ax.yaxis.set_ticks_position('left')
    ax.spines['left'].set_position(('data',-1.6))
    ax.xaxis.set_ticks_position('bottom')
    ax.spines['bottom'].set_position(('data',-0.08))
    ax.set_ylim(0,0.6)
    ax.set_xlim(-1.5,1.5)
    ax.set_xticks(np.arange(-1.5,1.501,0.5))
    ax.set_yticks(np.arange(0,0.601,0.2))
    ax.set_ylabel('Probability density',labelpad=10)

    ax.tick_params(labelsize=14)
    ax.xaxis.label.set_size(18)
    ax.yaxis.label.set_size(18)
    ax.xaxis.set_tick_params(width=2)
    ax.yaxis.set_tick_params(width=2)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
    #添加箱线图----------------------------
    pos = ax.get_position()
    # 在主图正下方建立 inset，sharex=ax 表示共用 x 轴
    ax1 = fig.add_axes([pos.x0, pos.y0 - 0.11, pos.width, 0.12],sharex=ax)
    ax1.axis('off')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['left'].set_visible(False)
    ax1.tick_params(bottom=False,top=False, left=False, right=False)#隐藏刻度线
    #ax.spines['bottom'].set_visible(False)
    bplot=ax1.boxplot(gc.nanravel(df_delta[i]),
                      vert=False,
                      whis=(10,90),                
                      widths=0.3,
                      patch_artist=True,
                      showmeans=False,
                      meanprops = {'marker':'o','markerfacecolor':cl[i],"markeredgecolor":cl[i],"markersize":15,"alpha":0}, # 设置均值点的属性，点的形状、填充色
                      medianprops={'linewidth':'0.5',"color":cl[i],"alpha":0},
                      boxprops={"facecolor": 'none', "edgecolor": cl[i],"linewidth":2,"alpha":1},
                      capprops=None,
                      whiskerprops={'linewidth':'2','color':cl[i]},
                      showcaps=False,# 是否显示箱线图顶端和末端的两条线，默认显示；
                      showfliers = False,
                      flierprops = {'marker':'+','markersize':'3','markeredgecolor':cl[i],'color':cl[i]})
    #rect = ax.patch
    #rect.set_facecolor((202/255,233/255,246/255))
#fig.savefig("E:/iCloudDrive/博后工作/文章/Elephant_disturb/Fig/reviseV2/Fig2_a-b.tif", dpi=600, bbox_inches = 'tight')

In [ ]:
#['treeH','tc_planet','lst']
#----------protect-----------
dt_protec=[delta_veg_protec[0],delta_veg_protec[1],delta_veg_protec[2]]
dd=np.copy(dt_protec)
dd[~np.isnan(dd)]=1
cc_mask_protec=np.ones((260,316))
for i in range(3):
    cc_mask_protec=cc_mask_protec*dd[i]
delta_dd_protec=np.copy(dt_protec)
for i in range(3):
    delta_dd_protec[i]=dt_protec[i]*cc_mask_protec
    print(np.count_nonzero(~np.isnan(delta_dd_protec[i])))

In [ ]:
#----------unprotect-----------
dt_unprotec=[delta_veg_unprotec[0],delta_veg_unprotec[1],delta_veg_unprotec[2]]
dd=np.copy(dt_unprotec)
dd[~np.isnan(dd)]=1
cc_mask_unprotec=np.ones((260,316))
for i in range(3):
    cc_mask_unprotec=cc_mask_unprotec*dd[i]
delta_dd_unprotec=np.copy(dt_unprotec)
for i in range(3):
    delta_dd_unprotec[i]=dt_unprotec[i]*cc_mask_unprotec
    print(np.count_nonzero(~np.isnan(delta_dd_unprotec[i])))

In [ ]:
#['treeH','tc_planet','lst']
pd_delta=[]
for i in range(3):
    dt=np.append(kill_nan(delta_dd_protec[i]),kill_nan(delta_dd_unprotec[i]))
    pd_delta.append(dt)
    print(len(dt))

In [ ]:
del_tree=[pd_delta[0],pd_delta[1]]
del_lst=pd_delta[2]

In [ ]:
import scipy
import statsmodels.api as sm
from scipy.stats import gaussian_kde
from sklearn.metrics import r2_score
from matplotlib.gridspec import GridSpec     # 利用网格确定图形的位置

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
fig = plt.figure(figsize=(12,4)) ##width, height default(8,6)
fig.subplots_adjust(wspace=0.3,hspace=0, left=None, bottom=None, right=None, top=None)
font = {'family': 'sans-serif',
        'sans-serif': 'Arial',
        'weight': 'normal',
        'size': 14}
plt.rc('font', **font)  # pass in the font dict as kwargs

cl=['red','blue']
mark=['*','.']
x_s1=np.arange(-10,20)
x_s2=np.arange(-50,75)
x_s=[x_s1,x_s2]
#cl=[(23/255,177/255,235/255),(236/255,25/255,32/255)]
cl1 = np.array([(82,160,134),(244,132,102)])
cl=cl1/255

for i in range(2):
    ax=fig.add_subplot(1,2,i+1)
    xy=np.vstack([del_tree[i],del_lst])
    z=gaussian_kde(xy)(xy)
    idx = z.argsort()
    x, y, z = np.array(del_tree[i])[idx], np.array(del_lst)[idx], z[idx]
    im=ax.scatter(del_tree[i],del_lst,c=z,alpha=0.5,s=30,cmap='Greys')
    # === 新增：statsmodels 线性回归 + 95% CI ===
    X = np.asarray(del_tree[i]).reshape(-1, 1)
    y = np.asarray(del_lst)
    X_sm = sm.add_constant(X)                # 加常数项
    model = sm.OLS(y, X_sm).fit()
    # 斜率和 95% CI
    params = model.params          # [intercept, slope]
    slope = params[1]
    ci_all = model.conf_int(alpha=0.05)
    ci_slope = ci_all[1]           # 斜率那一行

    print(f"Panel {['c','d'][i]}: slope = {slope:.3f}, 95% CI [{ci_slope[0]:.3f}, {ci_slope[1]:.3f}]")

    # 在当前面板的绘图范围上做平滑网格，避免超出轴范围
    if i == 0:
        xg = x_s1
    else:
        xg = x_s2
    xg = np.asarray(xg, dtype=float)
    Xg_sm = sm.add_constant(xg.reshape(-1, 1))
    pred = model.get_prediction(Xg_sm)
    yhat = pred.predicted_mean
    ci = pred.conf_int(alpha=0.05) 
    # 回归线与95%置信带
    ax.plot(xg, yhat, color='red', lw=1.5, zorder=3, label='OLS fit')
    ax.fill_between(xg, ci[:,0], ci[:,1], color='deepskyblue', alpha=0.6, linewidth=1, zorder=2, label='95% CI')
    #ax.text(0.8,0.88,'p = ' '{:.3f}'.format(p_value), transform = ax.transAxes,color=cl[i])
    cor = np.corrcoef(del_tree[i], del_lst)[0,1]
    ax.text(0.75, 0.8, 'R\u00b2 = {:.2f}\np<0.001'.format(cor*cor),transform=ax.transAxes, color='red')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(-9,6.01)
    ax.set_yticks(np.arange(-9,6.01,3))
    if i ==0:
        ax.set_xlim(-10,20.01)
        ax.set_xticks(np.arange(-10,20.01,10))
        #ax.spines['left'].set_visible(False)
        #ax.axes.yaxis.set_visible(False)
        ax.set_xlabel('$Δ$Tree height (m)',size=18,labelpad=15)
        ax.set_ylabel('$Δ$LST (°C)',size=18,labelpad=5)
        ax=plt.gca()
        ax.yaxis.set_ticks_position('left')
        ax.spines['left'].set_position(('data',-11))
        ax.spines['left'].set_linewidth(2);####设置左边坐标轴的粗细
        #ax.set_xlim(-10,20)
    if i==1:
        ax.set_xlim(-50,75.01)
        ax.set_xticks(np.arange(-50,75.01,25))
        #ax.spines['left'].set_visible(False)
        ax.set_xlabel('$Δ$Tree cover (%)',size=18,labelpad=15)
        ax.set_ylabel('$Δ$LST (°C)',size=18,labelpad=5)
        ax=plt.gca()
        ax.yaxis.set_ticks_position('left')
        ax.spines['left'].set_position(('data',-54))
        ax.spines['left'].set_linewidth(2);####设置左边坐标轴的粗细
        
        position=fig.add_axes([0.35,-0.15,0.3,0.05])#左、底、宽、高
        cbar = fig.colorbar(im,orientation='horizontal',cax=position)#,ticks=np.linspace(0, 1.5, 4)
        cbar.set_ticks(np.arange(0,0.031,0.01))
        cbar.ax.tick_params(labelsize=14)
        cbar.set_label(label='Data density',size=18)
        
    ax.xaxis.set_tick_params(width=2,labelsize=14,pad=5)
    ax.yaxis.set_tick_params(width=2,labelsize=14,pad=5)
    ax.spines['bottom'].set_linewidth(2);####设置左边坐标轴的粗细
#fig.savefig("E:/iCloudDrive/博后工作/文章/Elephant_disturb/Fig/reviseV2/Fig2_e-f.tif", dpi=600, bbox_inches = 'tight')

## Fig. 4

In [ ]:
#res & res-rerduc----------
#['ndvi_res','csif_res']
delta_res_protec1=np.load(spei_path+'delta_res_protect_250613.npy',allow_pickle=True)#(2,260,316)
delta_res_unprotec1=np.load(spei_path+'delta_res_unprotect_250613.npy',allow_pickle=True)#(2,260,316)
delta_res_reduc_protec1=np.load(spei_path+'delta_res_reduc_protect_250613.npy',allow_pickle=True)#(2,260,316)
delta_res_reduc_unprotec1=np.load(spei_path+'delta_res_reduc_unprotect_250613.npy',allow_pickle=True)#(2,260,316)
mm_protec=np.load(spei_path+'mask_reduc_protect_250613.npy',allow_pickle=True)
mm_unprotec=np.load(spei_path+'mask_reduc_unprotect_250613.npy',allow_pickle=True)

In [ ]:
def geo_mask(dem1,dem2,slo1,slo2):
    # check the Δ dem and Δ slope distribution
    dem_differ=dem1-dem2
    slo_differ=slo1-slo2

    dem_differ[dem_differ<-200]=np.nan
    dem_differ[dem_differ>200]=np.nan
    dem_differ[~np.isnan(dem_differ)]=1
    slo_differ[slo_differ<-10]=np.nan
    slo_differ[slo_differ>10]=np.nan
    slo_differ[~np.isnan(slo_differ)]=1
    # 每一层数据位置对应,建立mask
    dd_mask=np.ones((260,316))*dem_differ*slo_differ
    return dd_mask
#-------------------
dem_protec1=np.load(spei_path+'Vegetation_structure_protected_DEAA_dem_0.25deg_25613.npy',allow_pickle=True)
dem_protec2=np.load(spei_path+'Vegetation_structure_protected_DAA_dem_0.25deg_25613.npy',allow_pickle=True)
slope_protec1=np.load(spei_path+'Vegetation_structure_protected_DEAA_slope_0.25deg_25613.npy',allow_pickle=True)
slope_protec2=np.load(spei_path+'Vegetation_structure_protected_DAA_slope_0.25deg_25613.npy',allow_pickle=True)
dd_mask_protec=geo_mask(dem_protec1,dem_protec2,slope_protec1,slope_protec2)

dem_non_protec1=np.load(spei_path+'Vegetation_structure_non-protected_DEAA_dem_0.25deg_25613.npy',allow_pickle=True)
dem_non_protec2=np.load(spei_path+'Vegetation_structure_non-protected_DAA_dem_0.25deg_25613.npy',allow_pickle=True)
slope_non_protec1=np.load(spei_path+'Vegetation_structure_non-protected_DEAA_slope_0.25deg_25613.npy',allow_pickle=True)
slope_non_protec2=np.load(spei_path+'Vegetation_structure_non-protected_DAA_slope_0.25deg_25613.npy',allow_pickle=True)
dd_mask_non_protec=geo_mask(dem_non_protec1,dem_non_protec2,slope_non_protec1,slope_non_protec2)

In [ ]:
res_name=['ndvi_res','csif_res']
d_DAA_protec=[]
d_DAA_unprotec=[]
for i in range(2):
    f2_protec=np.load(spei_path+'Vegetation_structure_protected_DAA_{}_0.25deg_25613.npy'.format(res_name[i]),allow_pickle=True)*dd_mask_protec
    f2_unprotec=np.load(spei_path+'Vegetation_structure_non-protected_DAA_{}_0.25deg_25613.npy'.format(res_name[i]),allow_pickle=True)*dd_mask_non_protec
    d_DAA_protec.append(f2_protec)
    d_DAA_unprotec.append(f2_unprotec)

In [ ]:
relative_delta_protec=[]
relative_delta_unprotec=[]
for i in range(2):
    #protec--------------
    pre=(delta_res_protec1[i]/(1-d_DAA_protec[i]))*100
    pre_a=np.array(kill_nan(pre*mm_protec[i]))#会有一些1000%的异常点，剔除掉
    pre_a=np.delete(pre_a,np.where((pre_a<-1000)|(pre_a>1000)))#连接符“|”表示or，“&”表示and
    relative_delta_protec.append(pre_a)
    #unprotec---------------
    pre=(delta_res_unprotec1[i]/(1-d_DAA_unprotec[i]))*100
    pre_a=np.array(kill_nan(pre*mm_unprotec[i]))
    pre_a=np.delete(pre_a,np.where((pre_a<-1000)|(pre_a>1000)))#连接符“|”表示or，“&”表示and
    relative_delta_unprotec.append(pre_a)

np.mean(relative_delta_protec[0]),np.mean(relative_delta_protec[1]),np.mean(relative_delta_unprotec[0]),np.mean(relative_delta_unprotec[1])

In [ ]:
def kill_nan(dt):
    a=dt.ravel()
    a=a[~np.isnan(a)]
    a=list(a)
    return a
# Consider drought induced vegetation reduction
#pattern fig
#average pattens of protected and un protected
delta_ndvi = np.nanmean(np.stack((delta_res_reduc_protec1[0], delta_res_reduc_unprotec1[0])), axis=0)
delta_sif = np.nanmean(np.stack((delta_res_reduc_protec1[1], delta_res_reduc_unprotec1[1])), axis=0)
delta=[delta_ndvi,delta_sif]
#pdf resistance
res_delta_ndvi=np.append(kill_nan(delta_res_reduc_protec1[0]),kill_nan(delta_res_reduc_unprotec1[0]))
res_delta_sif=np.append(kill_nan(delta_res_reduc_protec1[1]),kill_nan(delta_res_reduc_unprotec1[1]))
res_delta=[res_delta_ndvi,res_delta_sif]
#boxplot relative resistance
relative_delta_ndvi=np.append(kill_nan(relative_delta_protec[0]),kill_nan(relative_delta_unprotec[0]))
relative_delta_sif=np.append(kill_nan(relative_delta_protec[1]),kill_nan(relative_delta_unprotec[1]))
relative_delta=[relative_delta_ndvi,relative_delta_sif]

In [ ]:
protec = read_img(drv_path + "Protected_area_AF.tif")[0]
print(protec.min(), protec.max())
protec = protec.astype(np.float32)
protec[protec == 0] = np.nan
protec.shape

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib import colors
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
font = {'family': 'sans-serif',
        'sans-serif': 'Helvetica',
        'weight': 'normal',
        'size': 16}

lon1 = -26
lon2 = 53
lat1 = -40
lat2 = 17
#lat2 = 17

left, bottom, width, hegiht= 0, 0.05, 0.5,1
left1, bottom1, width1, hegiht1= 0, -0.45, 0.5,1
loc=[[left, bottom, width, hegiht],[left1, bottom1, width1, hegiht1]]
left2, bottom2, width2, hegiht2= 0.65, 0.5, 0.3,0.3
left3, bottom3, width3, hegiht3= 0.65, -0.14, 0.3,0.4
loc1=[[left2, bottom2, width2, hegiht2],[left3, bottom3, width3, hegiht3]]


cl= np.array([(0,0,255),(159,159,255)])
cl=cl/255
cl1= np.array([(0,0,255),(109,109,255)])
cl1=cl1/255
res_name=['ndvi_res','csif_res']

fig = plt.figure(figsize=(12,8)) ##width, height default(8,6)
fig.text(0.02, 0.8, 'a', fontweight='bold', fontsize=24, fontname='Arial')
fig.text(0.55, 0.8, 'b', fontweight='bold', fontsize=24, fontname='Arial')
fig.text(0.02, 0.3, 'c', fontweight='bold', fontsize=24, fontname='Arial')
fig.text(0.55, 0.3, 'd', fontweight='bold', fontsize=24, fontname='Arial')
label=['$\Delta$ Resistance']
text=['a','b']
cmap = colors.ListedColormap([(197/255, 221/255, 197/255)])  # 只为值1定义颜色
bounds = [0.5, 1.5]                      # 把值1映射到这个范围
norm1 = colors.BoundaryNorm(bounds, cmap.N)

for i in range(2):
    ax = fig.add_subplot(loc[i])
    #patterns------------------------------------------------------------------
    m = Basemap(llcrnrlon=lon1,llcrnrlat=lat1,urcrnrlon=lon2,urcrnrlat=lat2,ax=ax)
    m.drawlsmask(land_color='whitesmoke',ocean_color='none',lakes=True)
    m.drawcountries(linewidth=0.30, color='dimgrey')
    norm = mcolors.TwoSlopeNorm(vmin=-0.05, vmax = 0.05, vcenter=0)
    im = m.imshow((delta[i])[32:],cmap ='bwr_r',origin='upper',norm=norm,zorder=2)
    im1 = m.imshow((protec)[800:],cmap =cmap,origin='upper',norm=norm1,zorder=1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)

    position=fig.add_axes([0.12,-0.2,0.3,0.01])#左、底、宽、高
    cb=fig.colorbar(im,ax=ax,extend='both',shrink=0.3,pad=0.01,orientation='horizontal',cax=position)#orientation='horizontal',
    cb.set_label(label[0],fontsize=18,labelpad=10)
    cb.ax.tick_params(labelsize=16) 
    cb.set_ticks(np.arange(-0.05,0.051,0.05)) 
    cb.outline.set_visible(False)
    # pdf--------------------------------------------------------------
    ax1=fig.add_subplot(loc1[0])
    x=np.linspace(-1,1,1000)
    mean=np.nanmean(res_delta[i])
    delta_rav=gc.nanravel(res_delta[i])
    kenal=gaussian_kde(delta_rav)
    z=kenal.evaluate(x)
    z_mean=kenal.evaluate(mean)
    print(mean,z_mean)

    xlabel=['$\Delta$Resistance','$\Delta$Resistance_SIF']
    ax1.plot(x,z,lw=3,color=cl[i])
    ax1.vlines(mean,0,z_mean,lw=2,ls='--',color=cl[i])
    ax1.vlines(0,0,25,lw=2,ls='--',color='black')
    ax1.spines['right'].set_visible(False)
    ax1.spines['top'].set_visible(False)
    p=kstest(df_delta[i], 'norm')[1]
    print("delta res, mean=",mean,"p=",p, "N=", len(delta_rav))
    print("delta res .s.e",stats.sem(delta_rav))
    if p<0.001:
        ax1.text(0.07,0.85-i*0.1, 'p<0.001', fontsize=14,transform = ax1.transAxes,color=cl[i])
    elif p<0.01:
        ax1.text(0.07,0.85-i*0.1, 'p<0.01', fontsize=14,transform = ax1.transAxes,color=cl[i])
    elif p<0.05:
        ax1.text(0.07,0.85-i*0.1, 'p<0.05', fontsize=14,transform = ax1.transAxes,color=cl[i])
    else:
        ax1.text(0.07,0.85-i*0.1, 'p={:.2f}'.format(p), fontsize=14,transform = ax1.transAxes,color=cl[i]) 
    
    ax1=plt.gca()
    ax1.yaxis.set_ticks_position('left')
    ax1.spines['left'].set_position(('data',-0.055))
    ax1.xaxis.set_ticks_position('bottom')
    ax1.spines['bottom'].set_position(('data',-4.5))
    ax1.set_ylim(0,18)
    ax1.set_xlim(-0.05,0.05)
    ax1.set_xticks(np.arange(-0.05,0.051,0.05))
    ax1.set_yticks(np.arange(0,20.1,10))
    ax1.set_ylabel('Probability density',labelpad=15)
    ax1.set_xlabel(xlabel[0],labelpad=15)
    ax1.tick_params(labelsize=16)
    ax1.xaxis.label.set_size(18)
    ax1.yaxis.label.set_size(18)
    ax1.xaxis.set_tick_params(width=2)
    ax1.yaxis.set_tick_params(width=2)
    ax1.spines['left'].set_linewidth(2)
    ax1.spines['bottom'].set_linewidth(2)
    ax1.set(facecolor='none')
    pos = ax1.get_position()
    if i ==0:
        ax2 = fig.add_axes([pos.x0, pos.y0 - 0.05, pos.width, 0.06],sharex=ax1)
        #ax2=fig.add_axes([0.7,0.35,0.2,0.06])#左，底，宽，高
        ax2.text(-0.0,0.325, "NDVI", fontsize=14,transform = ax2.transAxes,color=cl[i])#transform = ax1.transAxes,
    else:
        ax2 = fig.add_axes([pos.x0, pos.y0 - 0.08, pos.width, 0.06],sharex=ax1)
        #ax2=fig.add_axes([0.7,0.32,0.2,0.06])#左，底，宽，高
        ax2.text(-0.0,0.32, "SIF", fontsize=14,transform = ax2.transAxes,color=cl[i])#
    ax2.axis('off')
    ax2.set_xlim(-0.05,0.05)
    ax2.set_xticks(np.arange(-0.05,0.0501,0.05))
    ax2.spines['top'].set_visible(False)
    ax2.spines['bottom'].set_visible(True)
    bplot=ax2.boxplot(res_delta[i],
                      vert=False,
                      whis=(10,90),
                      widths=0.3,
                      patch_artist=True,
                      showmeans=False,
                      meanprops = {'marker':'o','markerfacecolor':'C0',"markeredgecolor":'C0',"markersize":15,"alpha":0}, # 设置均值点的属性，点的形状、填充色
                      medianprops={'linewidth':'0.5',"color":'C0',"alpha":0},
                      boxprops={"facecolor": 'none', "edgecolor": cl[i],"linewidth":2,"alpha":1},
                      capprops=None,
                      whiskerprops={'linewidth':'2','color':cl[i]},
                      showcaps=False,# 是否显示箱线图顶端和末端的两条线，默认显示
                      showfliers = False,
                      flierprops = {'marker':'+','markersize':'3','markeredgecolor':cl[i],'color':cl[i]})
    ax2.set(facecolor='none')
    #boxplot-------------------------------------------
    ax3=fig.add_subplot(loc1[1])
    ps=[[1],[1.5]]
    x1=[1,1.5]
    x2=[1.25]
    for i in range(2):
        ax3.boxplot(relative_delta[i],
                    vert=True,
                    whis=(10,90),                
                    widths=0.25,
                    patch_artist=True,
                    showmeans=False,
                    meanprops = {'marker':'o','markerfacecolor':'blue',"markeredgecolor":'blue',"markersize":15,"alpha":0}, # 设置均值点的属性，点的形状、填充色
                    medianprops={'linewidth':'2',"color":cl[i],"alpha":1},
                    boxprops={"facecolor": cl[i], "edgecolor": cl[i],"linewidth":2,"alpha":0.3},
                    capprops=None,
                    whiskerprops={'linewidth':'2','color':cl[i]},
                    showcaps=False,# 是否显示箱线图顶端和末端的两条线，默认显示
                    showfliers = False,
                    flierprops = {'marker':'+','markersize':'3','markeredgecolor':cl[i],'color':cl[i]},
                    positions=ps[i])
        patch=mpatches.Patch(color=cl[i],label='Short')
        ax3.scatter(x1[i],np.mean(relative_delta[i]),color= cl[i],s=130)
        ax3.hlines(0,0.75,1.75,lw=2,ls='--',color='black')
        p=kstest(relative_delta[i], 'norm')[1]
        print("relative res reduction, mean=", np.mean(relative_delta[i]), "p=", p, "N=", len(delta_rav))
        print("relative res reduction .s.e",stats.sem(relative_delta[i]))
        if p<0.001:
            ax3.text(0.07,0.85-i*0.08, 'p<0.001', fontsize=14,transform = ax3.transAxes,color=cl[i])
        elif p<0.01:
            ax3.text(0.07,0.85-i*0.08, 'p<0.01', fontsize=14,transform = ax3.transAxes,color=cl[i])
        elif p<0.05:
            ax3.text(0.07,0.85-i*0.08, 'p<0.05', fontsize=14,transform = ax3.transAxes,color=cl[i])
        else:
            ax3.text(0.07,0.85-i*0.08, 'p={:.2f}'.format(p), fontsize=14,transform = ax3.transAxes,color=cl[i]) 
        #ax3.text(0.07,0.85-i*0.08, "p<0.001", fontsize=16,transform = ax3.transAxes,color=cl[i])
    labels = ['NDVI', 'SIF']
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)
    ax3.spines['bottom'].set_visible(False)
    ax3.set_xticks(x1,labels=labels,rotation=0)
    ax3.xaxis.set_tick_params(labelsize=18,pad=10)
    ax3.yaxis.set_tick_params(labelsize=15)
    ax3.tick_params(bottom=True,top=False, left=True, right=False)#隐藏刻度线
    #ax3.set_xlabel('$\Delta$Resistance',labelpad=10)

    ax3.set_ylabel('PAL (%)',size=18,labelpad=15)
    ax3.set_ylim(-150,200)
    ax3.set_yticks(np.arange(-150,201,50))

    ax3.xaxis.set_tick_params(width=2)
    ax3.spines['left'].set_linewidth(2)

fig.savefig("C:/Users/dess/iCloudDrive/博后工作/Draft/Elephant_disturb/Fig/Fig3_resistance patterns_main_250624.png", dpi=300, bbox_inches = 'tight')

## Fig. 5a-d

In [ ]:
def geo_mask(dem1,dem2,slo1,slo2):
    # check the Δ dem and Δ slope distribution
    dem_differ=dem1-dem2
    slo_differ=slo1-slo2

    dem_differ[dem_differ<-200]=np.nan
    dem_differ[dem_differ>200]=np.nan
    dem_differ[~np.isnan(dem_differ)]=1
    slo_differ[slo_differ<-10]=np.nan
    slo_differ[slo_differ>10]=np.nan
    slo_differ[~np.isnan(slo_differ)]=1
    # 每一层数据位置对应,建立mask
    dd_mask=np.ones((260,316))*dem_differ*slo_differ
    return dd_mask

dem_protec1=np.load(spei_path+'Vegetation_structure_protected_DEAA_dem_0.25deg_25613.npy',allow_pickle=True)
dem_protec2=np.load(spei_path+'Vegetation_structure_protected_DAA_dem_0.25deg_25613.npy',allow_pickle=True)
slope_protec1=np.load(spei_path+'Vegetation_structure_protected_DEAA_slope_0.25deg_25613.npy',allow_pickle=True)
slope_protec2=np.load(spei_path+'Vegetation_structure_protected_DAA_slope_0.25deg_25613.npy',allow_pickle=True)
dd_mask_protec=geo_mask(dem_protec1,dem_protec2,slope_protec1,slope_protec2)

dem_non_protec1=np.load(spei_path+'Vegetation_structure_non-protected_DEAA_dem_0.25deg_25613.npy',allow_pickle=True)
dem_non_protec2=np.load(spei_path+'Vegetation_structure_non-protected_DAA_dem_0.25deg_25613.npy',allow_pickle=True)
slope_non_protec1=np.load(spei_path+'Vegetation_structure_non-protected_DEAA_slope_0.25deg_25613.npy',allow_pickle=True)
slope_non_protec2=np.load(spei_path+'Vegetation_structure_non-protected_DAA_slope_0.25deg_25613.npy',allow_pickle=True)
dd_mask_non_protec=geo_mask(dem_non_protec1,dem_non_protec2,slope_non_protec1,slope_non_protec2)

In [ ]:
def del_nan(data,data1):
    aa=np.where(np.isnan(data.ravel()))
    bb=np.where(np.isnan(data1.ravel()))
    kk=np.concatenate((aa,bb),axis=1)
    yy=np.unique(kk)
    data=np.delete(data.ravel(),yy)
    data1=np.delete(data1.ravel(),yy)
    return data, data1
def kill_nan(dt):
    a=dt.ravel()
    a=a[~np.isnan(a)]
    a=list(a)
    return a
#######################resistance单独拿出来提取reduction######################
# Add mean paras
res_name=['ndvi_res','csif_res']
###------------protected--------------------------------------------------------
mm_protec=[]#干旱导致ndvi和sif下降的格点mask
delta_protec1=[]
delta_reduc_protec1=[]
pp_protec1=[]
pp_reduc_protec1=[]
for i in range(2):
    f1=np.load(spei_path+'Vegetation_structure_protected_DEAA_{}_0.25deg_25613.npy'.format(res_name[i]),allow_pickle=True)*dd_mask_protec
    f2=np.load(spei_path+'Vegetation_structure_protected_DAA_{}_0.25deg_25613.npy'.format(res_name[i]),allow_pickle=True)*dd_mask_protec
    #---paired sites-------
    ff=f1-f2
    delta_protec1.append(ff)
    tt=f2
    tt[tt>1]=np.nan
    tt[tt<=1]=1
    mm_protec.append(tt)
    delta_reduc_protec1.append(ff*tt)
    #---pvalue------
    ddt=del_nan(f1,f2)
    sta,p=stats.wilcoxon(ddt[0],ddt[1])
    pp_protec1.append(p)
    ddt_reduc=del_nan(f1*tt,f2*tt)
    sta,p_reduc=stats.wilcoxon(ddt_reduc[0],ddt_reduc[1])
    pp_reduc_protec1.append(p_reduc)

###------------unprotected--------------------------------------------------------
mm_unprotec=[]#干旱导致ndvi和sif下降的格点mask
delta_unprotec1=[]
delta_reduc_unprotec1=[]
pp_unprotec1=[]
pp_reduc_unprotec1=[]
for i in range(2):
    f1=np.load(spei_path+'Vegetation_structure_non-protected_DEAA_{}_0.25deg_25613.npy'.format(res_name[i]),allow_pickle=True)*dd_mask_non_protec
    f2=np.load(spei_path+'Vegetation_structure_non-protected_DAA_{}_0.25deg_25613.npy'.format(res_name[i]),allow_pickle=True)*dd_mask_non_protec
    #---paired sites-------
    ff=f1-f2
    delta_unprotec1.append(ff)
    tt=f2
    tt[tt>1]=np.nan
    tt[tt<=1]=1
    mm_unprotec.append(tt)
    delta_reduc_unprotec1.append(ff*tt)
    #---pvalue------
    ddt=del_nan(f1,f2)
    sta,p=stats.wilcoxon(ddt[0],ddt[1])
    pp_unprotec1.append(p)
    ddt_reduc=del_nan(f1*tt,f2*tt)
    sta,p_reduc=stats.wilcoxon(ddt_reduc[0],ddt_reduc[1])
    pp_reduc_unprotec1.append(p_reduc)

In [ ]:
res_name=['treeH','tc','lst','spei','et','albedo']
delta_protec=[]
pp_protec=[]
delta_NDVIreduc_protec=[]
delta_SIFreduc_protec=[]

delta_unprotec=[]
pp_unprotec=[]
delta_NDVIreduc_unprotec=[]
delta_SIFreduc_unprotec=[]
for i in range(6):
    #protected areas--------------------
    f1_protec=np.load(spei_path+'Vegetation_structure_protected_DEAA_{}_0.25deg_25613.npy'.format(res_name[i]),allow_pickle=True)*dd_mask_protec
    f2_protec=np.load(spei_path+'Vegetation_structure_protected_DAA_{}_0.25deg_25613.npy'.format(res_name[i]),allow_pickle=True)*dd_mask_protec
    if i==3:
        ff_protec=(f1_protec+f2_protec)/2
    else:
        ff_protec=f1_protec-f2_protec
    delta_protec.append(ff_protec)
    delta_NDVIreduc_protec.append(ff_protec*mm_protec[0])
    delta_SIFreduc_protec.append(ff_protec*mm_protec[1])
    #p_value, protec
    ddt_protec=del_nan(f1_protec,f2_protec)
    sta,p_protec=stats.wilcoxon(ddt_protec[0],ddt_protec[1])
    pp_protec.append(p_protec)
    #non-protected areas----------------
    f1_unprotec=np.load(spei_path+'Vegetation_structure_non-protected_DEAA_{}_0.25deg_25613.npy'.format(res_name[i]),allow_pickle=True)*dd_mask_non_protec
    f2_unprotec=np.load(spei_path+'Vegetation_structure_non-protected_DAA_{}_0.25deg_25613.npy'.format(res_name[i]),allow_pickle=True)*dd_mask_non_protec
    if i==3:
        ff_unprotec=(f1_unprotec+f2_unprotec)/2
    else:
        ff_unprotec=f1_unprotec-f2_unprotec
    delta_unprotec.append(ff_unprotec)  
    delta_NDVIreduc_unprotec.append(ff_unprotec*mm_unprotec[0])
    delta_SIFreduc_unprotec.append(ff_unprotec*mm_unprotec[1])
    #p_value, non-protec
    ddt_unprotec=del_nan(f1_unprotec,f2_unprotec)
    sta,p_unprotec=stats.wilcoxon(ddt_unprotec[0],ddt_unprotec[1])
    pp_unprotec.append(p_unprotec)

In [ ]:
###----------基于drought reduction-----------------
###----------protected areas---------------
#需要匹配所有变量对应格点
#res_name=['treeH','tc','lst','spei','ndvi_res','csif_res','et','albedo']
#需要resistance, th, tc, lst, spei
dt_NDVIreduc_protec=[delta_reduc_protec1[0],
                     delta_NDVIreduc_protec[0],
                     delta_NDVIreduc_protec[1],
                     delta_NDVIreduc_protec[2],
                     delta_NDVIreduc_protec[3]]
dt_SIFreduc_protec=[delta_reduc_protec1[1],
                     delta_SIFreduc_protec[0],
                     delta_SIFreduc_protec[1],
                     delta_SIFreduc_protec[2],
                     delta_SIFreduc_protec[3]]

dd_ndvi=np.copy(dt_NDVIreduc_protec)
dd_ndvi[~np.isnan(dd_ndvi)]=1
cc_mask_ndvi=np.ones((260,316))
dd_sif=np.copy(dt_SIFreduc_protec)
dd_sif[~np.isnan(dd_sif)]=1
cc_mask_sif=np.ones((260,316))
for i in range(5):
    cc_mask_ndvi=cc_mask_ndvi*dd_ndvi[i]
    cc_mask_sif=cc_mask_sif*dd_sif[i]

delta_dd_ndvi_reduc_protec=np.copy(dt_NDVIreduc_protec)
delta_dd_sif_reduc_protec=np.copy(dt_SIFreduc_protec)
for i in range(5):
    delta_dd_ndvi_reduc_protec[i]=dt_NDVIreduc_protec[i]*cc_mask_ndvi
    delta_dd_sif_reduc_protec[i]=dt_SIFreduc_protec[i]*cc_mask_sif
    print(np.count_nonzero(~np.isnan(delta_dd_ndvi_reduc_protec[i])))
    print(np.count_nonzero(~np.isnan(delta_dd_sif_reduc_protec[i])))

In [ ]:
###----------基于drought reduction-----------------
###----------unprotected areas---------------
dt_NDVIreduc_unprotec=[delta_reduc_unprotec1[0],
                     delta_NDVIreduc_unprotec[0],
                     delta_NDVIreduc_unprotec[1],
                     delta_NDVIreduc_unprotec[2],
                     delta_NDVIreduc_unprotec[3]]
dt_SIFreduc_unprotec=[delta_reduc_unprotec1[1],
                     delta_SIFreduc_unprotec[0],
                     delta_SIFreduc_unprotec[1],
                     delta_SIFreduc_unprotec[2],
                     delta_SIFreduc_unprotec[3]]

dd_ndvi=np.copy(dt_NDVIreduc_unprotec)
dd_ndvi[~np.isnan(dd_ndvi)]=1
cc_mask_ndvi=np.ones((260,316))
dd_sif=np.copy(dt_SIFreduc_unprotec)
dd_sif[~np.isnan(dd_sif)]=1
cc_mask_sif=np.ones((260,316))
for i in range(5):
    cc_mask_ndvi=cc_mask_ndvi*dd_ndvi[i]
    cc_mask_sif=cc_mask_sif*dd_sif[i]

delta_dd_ndvi_reduc_unprotec=np.copy(dt_NDVIreduc_unprotec)
delta_dd_sif_reduc_unprotec=np.copy(dt_SIFreduc_unprotec)
for i in range(5):
    delta_dd_ndvi_reduc_unprotec[i]=dt_NDVIreduc_unprotec[i]*cc_mask_ndvi
    delta_dd_sif_reduc_unprotec[i]=dt_SIFreduc_unprotec[i]*cc_mask_sif
    print(np.count_nonzero(~np.isnan(delta_dd_ndvi_reduc_unprotec[i])))
    print(np.count_nonzero(~np.isnan(delta_dd_sif_reduc_unprotec[i])))

In [ ]:
#resistance, th, tc, lst, spei
pd_delta_ndvi=[]
pd_delta_sif=[]
for i in range(5):
    dt_ndvi=np.append(kill_nan(delta_dd_ndvi_reduc_protec[i]),kill_nan(delta_dd_ndvi_reduc_unprotec[i]))
    pd_delta_ndvi.append(dt_ndvi)
    print(len(dt_ndvi))
    dt_sif=np.append(kill_nan(delta_dd_sif_reduc_protec[i]),kill_nan(delta_dd_sif_reduc_unprotec[i]))
    pd_delta_sif.append(dt_sif)
    print(len(dt_sif))

In [ ]:
pd_delta_ndvi=list(map(list, zip(*pd_delta_ndvi)))#list转置
df_ndvi=pd.DataFrame(pd_delta_ndvi,columns=['NDVI_res','TH','TC','LST','SPEI_m'])
df_ndvi.info()

In [ ]:
pd_delta_sif=list(map(list, zip(*pd_delta_sif)))#list转置
df_sif=pd.DataFrame(pd_delta_sif,columns=['SIF_res','TH','TC','LST','SPEI_m'])
df_sif.info()

In [ ]:
#回归残差求偏相关系数
import statsmodels.api as sm
from statsmodels.formula.api import ols
#control=['SPEI','Precipitation',['SPEI','Precipitation'],'dif_LST',['dif_LST','SPEI'],['dif_LST','Precipitation'],['dif_LST','SPEI','Precipitation']]
#控制变量LST
th_lst_ndvi=sm.OLS(df_ndvi['TH'],df_ndvi['LST']).fit()
tc_lst_ndvi=sm.OLS(df_ndvi['TC'],df_ndvi['LST']).fit()
NDVIres_lst=sm.OLS(df_ndvi['NDVI_res'],df_ndvi['LST']).fit()

In [ ]:
th_lst_sif=sm.OLS(df_sif['TH'],df_sif['LST']).fit()
tc_lst_sif=sm.OLS(df_sif['TC'],df_sif['LST']).fit()
SIFres_lst=sm.OLS(df_sif['SIF_res'],df_sif['LST']).fit()

In [ ]:
res_tree_ndvi=[df_ndvi['TH'],th_lst_ndvi.resid,df_ndvi['TC'],tc_lst_ndvi.resid]
res_tree_sif=[df_sif['TH'],th_lst_sif.resid,df_sif['TC'],tc_lst_sif.resid]
res_res=[df_ndvi['NDVI_res'],df_sif['SIF_res']]
res_redius=[NDVIres_lst.resid,SIFres_lst.resid]

In [ ]:
res_tree=[res_tree_ndvi,res_tree_sif]

In [ ]:
import scipy
import matplotlib.pyplot as plt
import pingouin as pg
import seaborn as sns
import matplotlib.colors as colors
from scipy.stats import gaussian_kde
from sklearn.metrics import r2_score
from matplotlib.gridspec import GridSpec     # 利用网格确定图形的位置
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
fig = plt.figure(figsize=(16,9)) ##width, height default(8,6)
fig.subplots_adjust(wspace=0.3,hspace=0.3, left=None, bottom=None, right=None, top=None)
font = {'family': 'sans-serif',
        'sans-serif': 'Arial',
        'weight': 'normal',
        'size': 16}
plt.rc('font', **font)  # pass in the font dict as kwargs

cl=['royalblue','forestgreen']
mark=['*','.']
lab_x=['$Δ$TH (m)','$Δ$TH|$Δ$LST (m)','$Δ$TC (%)','$Δ$TC|$Δ$LST (%)']
lab_y=['$Δ$Resistance','$Δ$Resistance|$Δ$LST','$Δ$Resistance','$Δ$Resistance|$Δ$LST']
def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    new_cmap = colors.LinearSegmentedColormap.from_list(
        "trunc({n},{a:.2f},{b:.2f})".format(n=cmap.name, a=minval, b=maxval),
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap
trunc_cmap = [truncate_colormap(plt.get_cmap("Blues"), 0.6, 1),truncate_colormap(plt.get_cmap("Greens"), 0.5, 0.9)]
cov=[[],'LST',[],'LST']
x_a=['TH','TH','TC','TC']
y_a=['NDVI_res','SIF_res']

for i in range(4):
    ax=fig.add_subplot(2,2,i+1)
    if i<=1:
        xx=np.linspace(-12,25,1000)
        ax.set_ylim(-0.2,0.2)
        ax.set_yticks(np.arange(-0.2,0.201,0.1))
        ax.set_xlim(-15,30)
        ax.set_xticks(np.arange(-15,30.1,15))
    else:
        xx=np.linspace(-40,60,1000)
        ax.set_ylim(-0.2,0.2)
        ax.set_yticks(np.arange(-0.2,0.201,0.1))
        ax.set_xlim(-50,75)
        ax.set_xticks(np.arange(-50,75.1,25))
    if i==0 or i==2:
        for j in range(2):
            kde_plot = sns.kdeplot(x=res_tree[j][i], y=res_res[j], cmap=trunc_cmap[j], alpha=0.5,fill=True,ax=ax,levels=30, thresh=0.04)
            poly=np.polyfit(x=res_tree[j][i], y=res_res[j],deg=1)#一元二次方程拟合,,w=count[i]
            y_value=np.polyval(poly,xx)
            print(i,f"slope={poly[0]}")
            #yfit=np.polyval(poly,res_res[j])
            if j==0:
                cor=df_ndvi.partial_corr(x=x_a[i],y=y_a[j],covar=cov[i],method='spearman').round(3)['r']
                p_value=df_ndvi.partial_corr(x=x_a[i],y=y_a[j],covar=cov[i],method='spearman').round(3)['p-val']
            else:
                cor=df_sif.partial_corr(x=x_a[i],y=y_a[j],covar=cov[i],method='spearman').round(3)['r']
                p_value=df_sif.partial_corr(x=x_a[i],y=y_a[j],covar=cov[i],method='spearman').round(3)['p-val']
            if p_value[0]<0.001:
                linestyle='-'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p<0.001'.format(cor[0]), transform = ax.transAxes,color=cl[j])
            elif p_value[0]<0.01:
                linestyle='-'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p<0.01'.format(cor[0]), transform = ax.transAxes,color=cl[j])
            elif p_value[0]<0.05:
                linestyle='-'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p<0.05'.format(cor[0]), transform = ax.transAxes,color=cl[j])
            elif p_value[0]>=0.05:
                linestyle='--'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p={:.2f}'.format(cor[0],p_value[0]), transform = ax.transAxes,color=cl[j])
            ax.plot(xx,y_value,color=cl[j],ls=linestyle,linewidth=2)
    if i==1 or i==3:
        for j in range(2):
            kde_plot1 = sns.kdeplot(x=res_tree[j][i], y=res_redius[j], cmap=trunc_cmap[j], alpha=0.5,fill=True,ax=ax,levels=30, thresh=0.04)
            poly1=np.polyfit(x=res_tree[j][i], y=res_redius[j],deg=1)#一元二次方程拟合,,w=count[i]
            y_value1=np.polyval(poly1,xx)
            print(i,f"slope={poly1[0]}")
            #yfit1=np.polyval(poly1,res_redius[j])
            if j==0:
                cor=df_ndvi.partial_corr(x=x_a[i],y=y_a[j],covar=cov[i],method='spearman').round(3)['r']
                p_value=df_ndvi.partial_corr(x=x_a[i],y=y_a[j],covar=cov[i],method='spearman').round(3)['p-val']
            else:
                cor=df_sif.partial_corr(x=x_a[i],y=y_a[j],covar=cov[i],method='spearman').round(3)['r']
                p_value=df_sif.partial_corr(x=x_a[i],y=y_a[j],covar=cov[i],method='spearman').round(3)['p-val']
            #cor1,p_value1 = scipy.stats.spearmanr(res_tree[i],res_redius[j])
            if p_value[0]<0.001:
                linestyle='-'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p<0.001'.format(cor[0]), transform = ax.transAxes,color=cl[j])
            elif p_value[0]<0.01:
                linestyle='-'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p<0.01'.format(cor[0]), transform = ax.transAxes,color=cl[j])
            elif p_value[0]<0.05:
                linestyle='-'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p<0.05'.format(cor[0]), transform = ax.transAxes,color=cl[j])
            elif p_value[0]>=0.05:
                linestyle='--'
                ax.text(0.6,0.8+0.08*j,'r = {:.2f}  p={:.2f}'.format(cor[0],p_value[0]), transform = ax.transAxes,color=cl[j])
            ax.plot(xx,y_value1,color=cl[j],ls=linestyle,linewidth=2)
    ax.set_xlabel(lab_x[i],size=18,labelpad=10)
    ax.set_ylabel(lab_y[i],size=18,labelpad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
#fig.savefig("E:/iCloudDrive/博后工作/文章/Elephant_disturb/Fig/ReviseV4/Fig4a_d.tif", dpi=600, bbox_inches = 'tight')

## Fig. 5 SEM coefficients

Fig. 5 SEM coefficients are not redrawn in this notebook.

To reproduce the SEM coefficients, load:

`Data/path_analysis_SEM_reduction_250616.csv`

Then run:

`R_sem_240308_update_250614.R`